# Notebook 3: ML Experiments

Prototype ML pipeline with **scikit-learn** locally to validate feature engineering,  
understand class distribution, and estimate expected accuracy before running Spark MLlib on cluster.

The production pipeline is in `scripts/stage3_ml.py` (PySpark MLlib on YARN).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

%matplotlib inline

DATA_PATH = "../data/sf_incidents.csv"
TOP_N = 10  # keep top-N categories, rest → "Other"

df = pd.read_csv(DATA_PATH, low_memory=False)
df["Incident Datetime"] = pd.to_datetime(df["Incident Datetime"], format="%Y/%m/%d %I:%M:%S %p")
print(f"Loaded {len(df):,} rows")

In [ ]:
# ── Feature engineering ────────────────────────────────────────────────────
df["hour"]         = df["Incident Datetime"].dt.hour
df["day_of_week"]  = df["Incident Datetime"].dt.dayofweek
df["month"]        = df["Incident Datetime"].dt.month
df["year"]         = df["Incident Datetime"].dt.year

df["Police District"] = df["Police District"].fillna("Unknown")
district_enc = LabelEncoder()
df["district_enc"] = district_enc.fit_transform(df["Police District"])

# Bucket rare categories → "Other"
top_cats = df["Incident Category"].value_counts().head(TOP_N).index
df["label_raw"] = df["Incident Category"].where(
    df["Incident Category"].isin(top_cats), other="Other"
)

label_enc = LabelEncoder()
df["label"] = label_enc.fit_transform(df["label_raw"])

print("Label distribution:")
df["label_raw"].value_counts()

In [ ]:
# ── Prepare ML arrays ──────────────────────────────────────────────────────
FEATURES = ["hour", "day_of_week", "month", "year",
            "Latitude", "Longitude", "district_enc"]

data = df[FEATURES + ["label"]].dropna()
X = data[FEATURES].values
y = data["label"].values

print(f"Dataset for ML: {X.shape[0]:,} rows, {X.shape[1]} features")
print(f"Classes: {len(np.unique(y))}")

# Use a 10% sample for fast local prototyping
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.10, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.3, random_state=42, stratify=y_sample
)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

In [ ]:
# ── Model 1: Random Forest ─────────────────────────────────────────────────
rf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
])
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1  = f1_score(y_test, rf_pred, average="weighted")
print(f"Random Forest — Accuracy: {rf_acc:.4f} | Weighted F1: {rf_f1:.4f}")

In [ ]:
# ── Model 2: Linear SVC ────────────────────────────────────────────────────
svc = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearSVC(C=0.1, max_iter=1000, random_state=42))
])
svc.fit(X_train, y_train)
svc_pred = svc.predict(X_test)

svc_acc = accuracy_score(y_test, svc_pred)
svc_f1  = f1_score(y_test, svc_pred, average="weighted")
print(f"Linear SVC     — Accuracy: {svc_acc:.4f} | Weighted F1: {svc_f1:.4f}")

In [ ]:
# ── Model 3: Naive Bayes ───────────────────────────────────────────────────
nb = Pipeline([
    ("scaler", MinMaxScaler()),  # NB requires non-negative features
    ("clf", MultinomialNB(alpha=1.0))
])
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)

nb_acc = accuracy_score(y_test, nb_pred)
nb_f1  = f1_score(y_test, nb_pred, average="weighted")
print(f"Naive Bayes    — Accuracy: {nb_acc:.4f} | Weighted F1: {nb_f1:.4f}")

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
results = pd.DataFrame([
    {"Model": "Random Forest", "Accuracy": rf_acc,  "Weighted F1": rf_f1},
    {"Model": "Linear SVC",    "Accuracy": svc_acc, "Weighted F1": svc_f1},
    {"Model": "Naive Bayes",   "Accuracy": nb_acc,  "Weighted F1": nb_f1},
]).set_index("Model").round(4)

print("\n=== Local prototype results (10% sample) ===")
print(results.to_string())
print("\nNote: Spark MLlib on full dataset will produce different (better) results.")
results

In [ ]:
# ── Feature importance (Random Forest) ────────────────────────────────────
feat_imp = pd.Series(
    rf.named_steps["clf"].feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

feat_imp.plot(kind="barh", color="steelblue", figsize=(8, 4))
plt.title("Random Forest — Feature Importances")
plt.tight_layout()
plt.savefig("../output/eda/plot_feature_importance.png", dpi=150)
plt.show()

In [ ]:
# ── Classification report for best model ──────────────────────────────────
best_pred = rf_pred  # change to svc_pred or nb_pred as needed
print(classification_report(
    y_test, best_pred,
    target_names=label_enc.classes_,
    zero_division=0
))